In [ ]:
!pip install pandas numpy matplotlib seaborn plotly networkx openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx

from google.colab import files

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

In [ ]:
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
df.isnull().sum()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.dtypes

In [ ]:
df.describe().T

In [ ]:
# Convert date columns
date_columns = [
    "Date of Birth",
    "Sales Date",
    "Order Date",
    "Ship Date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Remove duplicate records
df = df.drop_duplicates()

# Fill missing numeric values with median
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill missing text values
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

# Create useful date features
df["Sales Year"] = df["Order Date"].dt.year
df["Sales Month"] = df["Order Date"].dt.month
df["Sales Month Name"] = df["Order Date"].dt.strftime("%B")
df["Day of Week"] = df["Order Date"].dt.strftime("%A")

# Profit margin
df["Profit Margin"] = np.where(
    df["Sales"] != 0,
    (df["Profit"] / df["Sales"]) * 100,
    0
)

# Shipping duration
df["Shipping Days"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

print("✅ Data cleaning completed!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

In [ ]:
df.isnull().sum().sum()

In [ ]:
df.head()

In [ ]:
  # ==============================
# RETAIL SALES KPIs
# ==============================

total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_quantity = df["Quantity"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()
total_products = df["Product ID"].nunique()

average_order_value = total_sales / total_orders

print("==========================================")
print("        RETAIL SALES ANALYTICS")
print("==========================================")

print(f"💰 Total Sales       : ₹{total_sales:,.2f}")
print(f"📈 Total Profit      : ₹{total_profit:,.2f}")
print(f"📦 Total Quantity    : {total_quantity:,}")
print(f"🧾 Total Orders      : {total_orders:,}")
print(f"👥 Total Customers   : {total_customers:,}")
print(f"🛍️ Total Products    : {total_products:,}")
print(f"💵 Average Order     : ₹{average_order_value:,.2f}")

In [ ]:
import plotly.express as px

monthly_sales = (
    df.groupby("Sales Month Name")["Sales"]
    .sum()
    .reset_index()
)

month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

monthly_sales["Sales Month Name"] = pd.Categorical(
    monthly_sales["Sales Month Name"],
    categories=month_order,
    ordered=True
)

monthly_sales = monthly_sales.sort_values(
    "Sales Month Name"
)

fig = px.line(
    monthly_sales,
    x="Sales Month Name",
    y="Sales",
    markers=True,
    title="Monthly Sales Trend"
)

fig.show()

In [ ]:
category_sales = (
    df.groupby("Category of Goods")["Sales"]
    .sum()
    .reset_index()
    .sort_values("Sales", ascending=False)
)

fig = px.bar(
    category_sales,
    x="Category of Goods",
    y="Sales",
    title="Sales by Category",
    text_auto=".2s"
)

fig.show()

In [ ]:
category_profit = (
    df.groupby("Category of Goods")["Profit"]
    .sum()
    .reset_index()
    .sort_values("Profit", ascending=False)
)

fig = px.bar(
    category_profit,
    x="Category of Goods",
    y="Profit",
    title="Profit by Category",
    text_auto=".2s"
)

fig.show()

In [ ]:
region_sales = (
    df.groupby("Region")["Sales"]
    .sum()
    .reset_index()
    .sort_values("Sales", ascending=False)
)

fig = px.bar(
    region_sales,
    x="Region",
    y="Sales",
    title="Sales by Region",
    text_auto=".2s"
)

fig.show()

In [ ]:
top_products = (
    df.groupby("Product Name")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    top_products,
    x="Sales",
    y="Product Name",
    orientation="h",
    title="Top 10 Products by Sales",
    text_auto=".2s"
)

fig.show()

In [ ]:
sample_df = df.sample(
    min(5000, len(df)),
    random_state=42
)

fig = px.scatter(
    sample_df,
    x="Sales",
    y="Profit",
    color="Category of Goods",
    size="Quantity",
    hover_data=[
        "Product Name",
        "Region",
        "Segment"
    ],
    title="Sales vs Profit"
)

fig.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Create graph
G = nx.Graph()

# Use top 100 customers by number of orders
top_customers = (
    df["Customer ID"]
    .value_counts()
    .head(100)
    .index
)

graph_df = df[
    df["Customer ID"].isin(top_customers)
].copy()

# Add customer-product relationships
for _, row in graph_df.iterrows():

    customer = "Customer_" + str(row["Customer ID"])
    product = "Product_" + str(row["Product Name"])

    G.add_node(
        customer,
        type="Customer"
    )

    G.add_node(
        product,
        type="Product"
    )

    if G.has_edge(customer, product):

        G[customer][product]["weight"] += 1

    else:

        G.add_edge(
            customer,
            product,
            weight=1
        )

print("Graph created successfully!")
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

In [ ]:
degree = dict(G.degree())

degree_df = pd.DataFrame(
    degree.items(),
    columns=[
        "Node",
        "Connections"
    ]
)

degree_df = degree_df.sort_values(
    "Connections",
    ascending=False
)

print("Top connected nodes:")
display(degree_df.head(20))

In [ ]:
product_degree = []

for node in G.nodes():

    if G.nodes[node]["type"] == "Product":

        product_degree.append({
            "Product": node.replace(
                "Product_",
                ""
            ),
            "Connections": G.degree(node)
        })

product_degree_df = pd.DataFrame(
    product_degree
).sort_values(
    "Connections",
    ascending=False
)

print("Most connected products:")
display(product_degree_df.head(10))

In [ ]:
centrality = nx.degree_centrality(G)

centrality_df = pd.DataFrame(
    centrality.items(),
    columns=[
        "Node",
        "Centrality"
    ]
)

centrality_df = centrality_df.sort_values(
    "Centrality",
    ascending=False
)

display(centrality_df.head(20))

In [ ]:
# Take the most connected 50 nodes
top_nodes = (
    centrality_df
    .head(50)["Node"]
    .tolist()
)

H = G.subgraph(top_nodes).copy()

plt.figure(figsize=(18, 12))

pos = nx.spring_layout(
    H,
    seed=42,
    k=0.8
)

# Different node sizes based on degree
node_sizes = [
    300 + G.degree(node) * 80
    for node in H.nodes()
]

# Customer vs Product
node_colors = []

for node in H.nodes():

    if H.nodes[node]["type"] == "Customer":
        node_colors.append("skyblue")
    else:
        node_colors.append("orange")

nx.draw_networkx(
    H,
    pos,
    node_size=node_sizes,
    node_color=node_colors,
    with_labels=True,
    font_size=7,
    edge_color="gray"
)

plt.title(
    "Customer–Product Relationship Network"
)

plt.axis("off")

plt.show()

In [ ]:
# ============================================================
# PRODUCT CO-PURCHASE ANALYSIS
# ============================================================

from itertools import combinations
from collections import Counter

pair_counter = Counter()

# Get unique products for every order
order_products = (
    df.groupby("Order ID")["Product Name"]
    .apply(lambda x: list(set(x)))
)

# Count product pairs
for products in order_products:

    if len(products) >= 2:

        for pair in combinations(
            sorted(products),
            2
        ):
            pair_counter[pair] += 1


# Check whether product pairs exist
if len(pair_counter) == 0:

    print("⚠️ No multiple-product orders were found.")
    print("Creating an alternative product relationship graph...")

else:

    co_purchase_df = pd.DataFrame(
        [
            {
                "Product 1": pair[0],
                "Product 2": pair[1],
                "Purchase Count": count
            }
            for pair, count in pair_counter.items()
        ]
    )

    co_purchase_df = co_purchase_df.sort_values(
        "Purchase Count",
        ascending=False
    ).reset_index(drop=True)

    print("✅ Product co-purchase analysis completed!")

    display(
        co_purchase_df.head(20)
    )

In [ ]:
if len(pair_counter) > 0:

    product_graph = nx.Graph()

    for _, row in co_purchase_df.head(50).iterrows():

        product1 = row["Product 1"]
        product2 = row["Product 2"]
        weight = row["Purchase Count"]

        product_graph.add_edge(
            product1,
            product2,
            weight=weight
        )

    print(
        "Product graph nodes:",
        product_graph.number_of_nodes()
    )

    print(
        "Product graph edges:",
        product_graph.number_of_edges()
    )

In [ ]:
if len(pair_counter) > 0:

    plt.figure(figsize=(16, 12))

    pos = nx.spring_layout(
        product_graph,
        seed=42
    )

    edge_weights = [
        product_graph[u][v]["weight"] / 5
        for u, v in product_graph.edges()
    ]

    nx.draw_networkx(
        product_graph,
        pos,
        node_size=1200,
        with_labels=True,
        font_size=9,
        edge_color="gray",
        width=edge_weights
    )

    plt.title(
        "Product Co-Purchase Network"
    )

    plt.axis("off")

    plt.show()

else:

    print(
        "Product co-purchase graph cannot be created "
        "because the dataset has no orders containing "
        "multiple products."
    )

In [ ]:
!pip -q install streamlit pyngrok

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx

# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Retail Sales Analytics",
    page_icon="📊",
    layout="wide"
)

# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>

.main {
    background-color: #f5f7fb;
}

.metric-card {
    background-color: white;
    padding: 20px;
    border-radius: 12px;
    box-shadow: 0px 2px 8px rgba(0,0,0,0.08);
}

h1 {
    color: #1f2937;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# LOAD DATA
# ============================================================

@st.cache_data
def load_data():

    data = pd.read_csv(
        "retail_sales_cleaned.csv"
    )

    data["Order Date"] = pd.to_datetime(
        data["Order Date"],
        errors="coerce"
    )

    return data


df = load_data()


# ============================================================
# TITLE
# ============================================================

st.title(
    "📊 Retail Sales Analytics Dashboard"
)

st.markdown(
    "### Web-Based Graph Analytics for Retail Sales"
)

st.divider()


# ============================================================
# SIDEBAR FILTERS
# ============================================================

st.sidebar.header(
    "🔎 Dashboard Filters"
)


# Region filter
regions = sorted(
    df["Region"].dropna().unique()
)

selected_regions = st.sidebar.multiselect(
    "Region",
    regions,
    default=regions
)


# Category filter
categories = sorted(
    df["Category of Goods"].dropna().unique()
)

selected_categories = st.sidebar.multiselect(
    "Category",
    categories,
    default=categories
)


# Segment filter
segments = sorted(
    df["Segment"].dropna().unique()
)

selected_segments = st.sidebar.multiselect(
    "Customer Segment",
    segments,
    default=segments
)


# Filter data
filtered_df = df[
    (df["Region"].isin(selected_regions)) &
    (df["Category of Goods"].isin(selected_categories)) &
    (df["Segment"].isin(selected_segments))
]


# ============================================================
# KPI CALCULATIONS
# ============================================================

total_sales = filtered_df["Sales"].sum()

total_profit = filtered_df["Profit"].sum()

total_orders = filtered_df["Order ID"].nunique()

total_customers = filtered_df["Customer ID"].nunique()

total_quantity = filtered_df["Quantity"].sum()

avg_order_value = (
    total_sales / total_orders
    if total_orders > 0
    else 0
)


# ============================================================
# KPI CARDS
# ============================================================

col1, col2, col3, col4, col5 = st.columns(5)

col1.metric(
    "💰 Total Sales",
    f"₹{total_sales:,.0f}"
)

col2.metric(
    "📈 Total Profit",
    f"₹{total_profit:,.0f}"
)

col3.metric(
    "🧾 Orders",
    f"{total_orders:,}"
)

col4.metric(
    "👥 Customers",
    f"{total_customers:,}"
)

col5.metric(
    "📦 Quantity",
    f"{total_quantity:,}"
)


st.divider()


# ============================================================
# SALES TREND
# ============================================================

st.subheader("📈 Sales Trend")

monthly_sales = (
    filtered_df.dropna(subset=["Order Date"])
    .groupby(filtered_df["Order Date"].dt.strftime("%Y-%m"))["Sales"]
    .sum()
    .reset_index()
)

if len(monthly_sales) > 0:
    fig_sales = px.line(
        monthly_sales,
        x="Order Date",
        y="Sales",
        markers=True,
        title="Monthly Sales Trend"
    )

    fig_sales.update_layout(
        xaxis_title="Month",
        yaxis_title="Sales"
    )

    st.plotly_chart(fig_sales, use_container_width=True)

else:
    st.warning("No data available for Sales Trend")

# ============================================================
# CATEGORY + REGION
# ============================================================

col1, col2 = st.columns(2)


# -----------------------------
# Category
# -----------------------------

with col1:

    category_sales = (
        filtered_df
        .groupby(
            "Category of Goods"
        )["Sales"]
        .sum()
        .reset_index()
        .sort_values(
            "Sales",
            ascending=False
        )
    )

    fig_category = px.bar(
        category_sales,
        x="Category of Goods",
        y="Sales",
        title="Sales by Category"
    )

    st.plotly_chart(
        fig_category,
        use_container_width=True
    )


# -----------------------------
# Region
# -----------------------------

with col2:

    region_sales = (
        filtered_df
        .groupby(
            "Region"
        )["Sales"]
        .sum()
        .reset_index()
        .sort_values(
            "Sales",
            ascending=False
        )
    )

    fig_region = px.bar(
        region_sales,
        x="Region",
        y="Sales",
        title="Sales by Region"
    )

    st.plotly_chart(
        fig_region,
        use_container_width=True
    )


# ============================================================
# TOP PRODUCTS
# ============================================================

st.subheader(
    "🏆 Top 10 Products"
)

top_products = (
    filtered_df
    .groupby(
        "Product Name"
    )["Sales"]
    .sum()
    .sort_values(
        ascending=False
    )
    .head(10)
    .reset_index()
)

fig_products = px.bar(
    top_products,
    x="Sales",
    y="Product Name",
    orientation="h",
    title="Top Products by Sales"
)

st.plotly_chart(
    fig_products,
    use_container_width=True
)


# ============================================================
# SALES VS PROFIT
# ============================================================

st.subheader(
    "📊 Sales vs Profit"
)

sample = filtered_df.sample(
    min(5000, len(filtered_df)),
    random_state=42
)

fig_scatter = px.scatter(
    sample,
    x="Sales",
    y="Profit",
    color="Category of Goods",
    size="Quantity",
    hover_data=[
        "Product Name",
        "Region",
        "Segment"
    ],
    title="Sales vs Profit Relationship"
)

st.plotly_chart(
    fig_scatter,
    use_container_width=True
)


# ============================================================
# GRAPH ANALYTICS
# ============================================================

st.divider()

st.header(
    "🔗 Graph Analytics"
)

st.markdown(
    """
    The graph represents relationships between
    **customers and products**.
    """
)


# Use top 100 customers
top_customers = (
    filtered_df["Customer ID"]
    .value_counts()
    .head(100)
    .index
)

graph_df = filtered_df[
    filtered_df["Customer ID"]
    .isin(top_customers)
]


G = nx.Graph()


for _, row in graph_df.iterrows():

    customer = (
        "C_" +
        str(row["Customer ID"])
    )

    product = (
        "P_" +
        str(row["Product Name"])
    )

    G.add_node(
        customer,
        type="Customer"
    )

    G.add_node(
        product,
        type="Product"
    )

    if G.has_edge(
        customer,
        product
    ):

        G[customer][product]["weight"] += 1

    else:

        G.add_edge(
            customer,
            product,
            weight=1
        )


# ============================================================
# CENTRALITY
# ============================================================

centrality = nx.degree_centrality(G)

centrality_df = pd.DataFrame(
    centrality.items(),
    columns=[
        "Node",
        "Centrality"
    ]
)

centrality_df = (
    centrality_df
    .sort_values(
        "Centrality",
        ascending=False
    )
)


# Product nodes
product_nodes = [
    node
    for node in G.nodes()
    if G.nodes[node]["type"] == "Product"
]

product_centrality = (
    centrality_df[
        centrality_df["Node"]
        .isin(product_nodes)
    ]
    .copy()
)

product_centrality["Product"] = (
    product_centrality["Node"]
    .str.replace(
        "P_",
        "",
        regex=False
    )
)


# ============================================================
# GRAPH METRICS
# ============================================================

col1, col2, col3 = st.columns(3)

col1.metric(
    "Graph Nodes",
    G.number_of_nodes()
)

col2.metric(
    "Graph Connections",
    G.number_of_edges()
)

col3.metric(
    "Products Analyzed",
    len(product_nodes)
)


# ============================================================
# TOP CONNECTED PRODUCTS
# ============================================================

st.subheader(
    "⭐ Most Connected Products"
)

top_connected = (
    product_centrality
    .head(10)
)

fig_centrality = px.bar(
    top_connected,
    x="Centrality",
    y="Product",
    orientation="h",
    title="Product Network Centrality"
)

st.plotly_chart(
    fig_centrality,
    use_container_width=True
)


# ============================================================
# NETWORK GRAPH
# ============================================================

st.subheader(
    "🕸️ Customer–Product Network"
)

top_nodes = (
    centrality_df
    .head(40)["Node"]
    .tolist()
)

H = G.subgraph(
    top_nodes
).copy()

pos = nx.spring_layout(
    H,
    seed=42
)

edge_x = []
edge_y = []

for edge in H.edges():

    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]

    edge_x += [
        x0,
        x1,
        None
    ]

    edge_y += [
        y0,
        y1,
        None
    ]


edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode="lines",
    line=dict(
        width=1
    ),
    hoverinfo="none"
)


node_x = []
node_y = []
node_text = []

for node in H.nodes():

    x, y = pos[node]

    node_x.append(x)
    node_y.append(y)

    node_text.append(node)


node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers+text",
    text=node_text,
    textposition="top center",
    hoverinfo="text",
    marker=dict(
        size=15
    )
)


network_fig = go.Figure(
    data=[
        edge_trace,
        node_trace
    ]
)

network_fig.update_layout(
    title="Customer–Product Relationship Network",
    showlegend=False,
    xaxis=dict(
        showgrid=False,
        zeroline=False,
        showticklabels=False
    ),
    yaxis=dict(
        showgrid=False,
        zeroline=False,
        showticklabels=False
    )
)

st.plotly_chart(
    network_fig,
    use_container_width=True
)


# ============================================================
# DATA TABLE
# ============================================================

st.divider()

st.subheader(
    "📋 Sales Data"
)

st.dataframe(
    filtered_df.head(1000),
    use_container_width=True
)


# ============================================================
# DOWNLOAD
# ============================================================

csv = filtered_df.to_csv(
    index=False
)

st.download_button(
    label="⬇️ Download Filtered Data",
    data=csv,
    file_name="filtered_retail_sales.csv",
    mime="text/csv"
)

In [ ]:
# Save the cleaned dataframe
df.to_csv("retail_sales_cleaned.csv", index=False)

print("Dataset saved successfully!")
print("Rows:", len(df))
print("File exists:", os.path.exists("retail_sales_cleaned.csv"))

In [ ]:
!streamlit run app.py &> /content/streamlit.log &

In [ ]:
!cat /content/streamlit.log

In [ ]:
!pip install -q pyngrok

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!cloudflared tunnel --url http://localhost:8501

In [ ]:
# Fix missing Plotly Graph Objects import
with open("app.py", "r") as f:
    code = f.read()

if "import plotly.graph_objects as go" not in code:
    code = code.replace(
        "import plotly.express as px",
        "import plotly.express as px\nimport plotly.graph_objects as go"
    )

with open("app.py", "w") as f:
    f.write(code)

print("✅ app.py fixed!")
print("✅ Added: import plotly.graph_objects as go")

In [ ]:
!pkill -f "streamlit run app.py" || true

In [ ]:
!streamlit run app.py &> /content/streamlit.log &

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from itertools import combinations

# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Retail Sales Analytics",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>

.main {
    background-color: #0e1117;
}

.block-container {
    padding-top: 2rem;
    padding-bottom: 2rem;
}

.dashboard-title {
    font-size: 42px;
    font-weight: 800;
    margin-bottom: 5px;
}

.subtitle {
    font-size: 20px;
    color: #aaaaaa;
    margin-bottom: 30px;
}

.metric-card {
    background: linear-gradient(135deg, #1f2937, #111827);
    padding: 20px;
    border-radius: 15px;
    border: 1px solid #374151;
    min-height: 120px;
}

.section-title {
    font-size: 28px;
    font-weight: 700;
    margin-top: 35px;
    margin-bottom: 15px;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# LOAD DATA
# ============================================================

@st.cache_data
def load_data():

    df = pd.read_csv("retail_sales_cleaned.csv")

    # -----------------------------
    # Convert dates
    # -----------------------------

    date_columns = [
        "Date of Birth",
        "Sales Date",
        "Order Date",
        "Ship Date"
    ]

    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )

    # -----------------------------
    # Convert numeric columns
    # -----------------------------

    numeric_columns = [
        "Sales",
        "Quantity",
        "Discount",
        "Profit",
        "Postal Code"
    ]

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    # -----------------------------
    # Remove invalid rows
    # -----------------------------

    df = df.dropna(
        subset=["Sales", "Profit", "Quantity"]
    )

    # -----------------------------
    # Create date features
    # -----------------------------

    if "Order Date" in df.columns:

        df["Sales Year"] = df["Order Date"].dt.year

        df["Sales Month Number"] = (
            df["Order Date"].dt.month
        )

        df["Sales Month"] = (
            df["Order Date"].dt.strftime("%Y-%m")
        )

        df["Sales Month Name"] = (
            df["Order Date"].dt.strftime("%b")
        )

        df["Day of Week"] = (
            df["Order Date"].dt.day_name()
        )

    # -----------------------------
    # Profit margin
    # -----------------------------

    df["Profit Margin"] = np.where(
        df["Sales"] != 0,
        (df["Profit"] / df["Sales"]) * 100,
        0
    )

    return df


df = load_data()


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.markdown(
    "## 🔎 Dashboard Filters"
)

# Region
if "Region" in df.columns:

    regions = sorted(
        df["Region"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    selected_regions = st.sidebar.multiselect(
        "Region",
        regions,
        default=regions
    )

else:
    selected_regions = []


# Category
if "Category of Goods" in df.columns:

    categories = sorted(
        df["Category of Goods"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    selected_categories = st.sidebar.multiselect(
        "Category",
        categories,
        default=categories
    )

else:
    selected_categories = []


# Segment
if "Segment" in df.columns:

    segments = sorted(
        df["Segment"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    selected_segments = st.sidebar.multiselect(
        "Customer Segment",
        segments,
        default=segments
    )

else:
    selected_segments = []


# ============================================================
# FILTER DATA
# ============================================================

filtered_df = df.copy()

if selected_regions and "Region" in filtered_df.columns:

    filtered_df = filtered_df[
        filtered_df["Region"]
        .astype(str)
        .isin(selected_regions)
    ]

if selected_categories and "Category of Goods" in filtered_df.columns:

    filtered_df = filtered_df[
        filtered_df["Category of Goods"]
        .astype(str)
        .isin(selected_categories)
    ]

if selected_segments and "Segment" in filtered_df.columns:

    filtered_df = filtered_df[
        filtered_df["Segment"]
        .astype(str)
        .isin(selected_segments)
    ]


# ============================================================
# HEADER
# ============================================================

st.markdown(
    '<div class="dashboard-title">📊 Retail Sales Analytics Dashboard</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">Web-Based Graph Analytics for Retail Sales</div>',
    unsafe_allow_html=True
)

st.divider()


# ============================================================
# EMPTY DATA CHECK
# ============================================================

if filtered_df.empty:

    st.warning(
        "⚠️ No data matches the selected filters."
    )

    st.stop()


# ============================================================
# KPI SECTION
# ============================================================

total_sales = filtered_df["Sales"].sum()

total_profit = filtered_df["Profit"].sum()

total_quantity = filtered_df["Quantity"].sum()

total_orders = (
    filtered_df["Order ID"].nunique()
    if "Order ID" in filtered_df.columns
    else len(filtered_df)
)

total_customers = (
    filtered_df["Customer ID"].nunique()
    if "Customer ID" in filtered_df.columns
    else 0
)


col1, col2, col3, col4, col5 = st.columns(5)

with col1:

    st.metric(
        "💰 Total Sales",
        f"₹{total_sales:,.0f}"
    )

with col2:

    st.metric(
        "📈 Total Profit",
        f"₹{total_profit:,.0f}"
    )

with col3:

    st.metric(
        "🧾 Orders",
        f"{total_orders:,}"
    )

with col4:

    st.metric(
        "👥 Customers",
        f"{total_customers:,}"
    )

with col5:

    st.metric(
        "📦 Quantity",
        f"{total_quantity:,.0f}"
    )


# ============================================================
# SALES TREND
# ============================================================

st.markdown(
    '<div class="section-title">📈 Monthly Sales Trend</div>',
    unsafe_allow_html=True
)

if "Order Date" in filtered_df.columns:

    monthly_sales = (
        filtered_df
        .dropna(subset=["Order Date"])
        .groupby(
            pd.Grouper(
                key="Order Date",
                freq="MS"
            )
        )["Sales"]
        .sum()
        .reset_index()
        .sort_values("Order Date")
    )

    if not monthly_sales.empty:

        fig = px.line(
            monthly_sales,
            x="Order Date",
            y="Sales",
            markers=True,
            title="Monthly Sales Trend"
        )

        fig.update_layout(
            template="plotly_dark",
            height=450,
            xaxis_title="Month",
            yaxis_title="Sales (₹)",
            hovermode="x unified"
        )

        fig.update_traces(
            line_width=3
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    else:

        st.warning("No date data available for Sales Trend.")


# ============================================================
# CATEGORY ANALYSIS
# ============================================================

st.markdown(
    '<div class="section-title">🏷️ Sales by Category</div>',
    unsafe_allow_html=True
)

if "Category of Goods" in filtered_df.columns:

    category_sales = (
        filtered_df
        .groupby("Category of Goods")["Sales"]
        .sum()
        .reset_index()
        .sort_values(
            "Sales",
            ascending=False
        )
    )

    fig = px.bar(
        category_sales,
        x="Category of Goods",
        y="Sales",
        text_auto=".2s",
        title="Sales by Product Category"
    )

    fig.update_layout(
        template="plotly_dark",
        height=450,
        xaxis_title="Category",
        yaxis_title="Sales (₹)"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


# ============================================================
# PROFIT BY CATEGORY
# ============================================================

st.markdown(
    '<div class="section-title">💰 Profit by Category</div>',
    unsafe_allow_html=True
)

if "Category of Goods" in filtered_df.columns:

    category_profit = (
        filtered_df
        .groupby("Category of Goods")["Profit"]
        .sum()
        .reset_index()
        .sort_values(
            "Profit",
            ascending=False
        )
    )

    fig = px.bar(
        category_profit,
        x="Category of Goods",
        y="Profit",
        text_auto=".2s",
        title="Profit by Product Category"
    )

    fig.update_layout(
        template="plotly_dark",
        height=450,
        xaxis_title="Category",
        yaxis_title="Profit (₹)"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


# ============================================================
# REGION ANALYSIS
# ============================================================

st.markdown(
    '<div class="section-title">🌎 Sales by Region</div>',
    unsafe_allow_html=True
)

if "Region" in filtered_df.columns:

    region_sales = (
        filtered_df
        .groupby("Region")["Sales"]
        .sum()
        .reset_index()
        .sort_values(
            "Sales",
            ascending=False
        )
    )

    fig = px.bar(
        region_sales,
        x="Region",
        y="Sales",
        text_auto=".2s",
        title="Regional Sales Performance"
    )

    fig.update_layout(
        template="plotly_dark",
        height=450,
        xaxis_title="Region",
        yaxis_title="Sales (₹)"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


# ============================================================
# TOP PRODUCTS
# ============================================================

st.markdown(
    '<div class="section-title">🏆 Top 10 Products</div>',
    unsafe_allow_html=True
)

if "Product Name" in filtered_df.columns:

    top_products = (
        filtered_df
        .groupby("Product Name")["Sales"]
        .sum()
        .reset_index()
        .sort_values(
            "Sales",
            ascending=False
        )
        .head(10)
    )

    fig = px.bar(
        top_products,
        x="Sales",
        y="Product Name",
        orientation="h",
        text_auto=".2s",
        title="Top 10 Products by Sales"
    )

    fig.update_layout(
        template="plotly_dark",
        height=500,
        yaxis={
            "categoryorder": "total ascending"
        },
        xaxis_title="Sales (₹)",
        yaxis_title="Product"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


# ============================================================
# SALES VS PROFIT
# ============================================================

st.markdown(
    '<div class="section-title">📊 Sales vs Profit</div>',
    unsafe_allow_html=True
)

if "Category of Goods" in filtered_df.columns:

    sales_profit = (
        filtered_df
        .groupby("Category of Goods")
        [["Sales", "Profit"]]
        .sum()
        .reset_index()
    )

    fig = px.scatter(
        sales_profit,
        x="Sales",
        y="Profit",
        size="Sales",
        hover_name="Category of Goods",
        title="Sales vs Profit by Category"
    )

    fig.update_layout(
        template="plotly_dark",
        height=500,
        xaxis_title="Sales (₹)",
        yaxis_title="Profit (₹)"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


# ============================================================
# GRAPH ANALYTICS
# ============================================================

st.markdown(
    '<div class="section-title">🕸️ Customer–Product Network</div>',
    unsafe_allow_html=True
)

st.write(
    "Graph-based analysis showing relationships between customers and products."
)


if (
    "Customer ID" in filtered_df.columns
    and "Product ID" in filtered_df.columns
):

    # --------------------------------------------------------
    # Top customers
    # --------------------------------------------------------

    top_customers = (
        filtered_df["Customer ID"]
        .value_counts()
        .head(50)
        .index
    )

    network_df = filtered_df[
        filtered_df["Customer ID"].isin(top_customers)
    ].copy()

    # --------------------------------------------------------
    # Create graph
    # --------------------------------------------------------

    G = nx.Graph()

    for _, row in network_df.iterrows():

        customer = "C_" + str(row["Customer ID"])

        product = "P_" + str(row["Product ID"])

        if G.has_edge(customer, product):

            G[customer][product]["weight"] += 1

        else:

            G.add_edge(
                customer,
                product,
                weight=1
            )

    # --------------------------------------------------------
    # Limit graph size
    # --------------------------------------------------------

    if len(G.nodes()) > 0:

        degrees = dict(
            G.degree()
        )

        top_nodes = sorted(
            degrees,
            key=degrees.get,
            reverse=True
        )[:60]

        G = G.subgraph(
            top_nodes
        ).copy()

        # ----------------------------------------------------
        # Network layout
        # ----------------------------------------------------

        pos = nx.spring_layout(
            G,
            seed=42,
            k=0.7
        )

        edge_x = []
        edge_y = []

        for edge in G.edges():

            x0, y0 = pos[edge[0]]

            x1, y1 = pos[edge[1]]

            edge_x.extend(
                [x0, x1, None]
            )

            edge_y.extend(
                [y0, y1, None]
            )

        edge_trace = go.Scatter(
            x=edge_x,
            y=edge_y,
            mode="lines",
            line=dict(
                width=1
            ),
            hoverinfo="none"
        )

        # ----------------------------------------------------
        # Nodes
        # ----------------------------------------------------

        node_x = []
        node_y = []
        node_text = []

        for node in G.nodes():

            x, y = pos[node]

            node_x.append(x)

            node_y.append(y)

            node_text.append(
                f"{node}<br>Connections: {G.degree(node)}"
            )

        node_trace = go.Scatter(
            x=node_x,
            y=node_y,
            mode="markers+text",
            text=[
                node.split("_", 1)[1]
                for node in G.nodes()
            ],
            textposition="top center",
            hovertext=node_text,
            hoverinfo="text",
            marker=dict(
                size=16
            )
        )

        # ----------------------------------------------------
        # Figure
        # ----------------------------------------------------

        fig = go.Figure(
            data=[
                edge_trace,
                node_trace
            ]
        )

        fig.update_layout(
            template="plotly_dark",
            height=650,
            title="Customer–Product Relationship Graph",
            showlegend=False,
            xaxis=dict(
                showgrid=False,
                zeroline=False,
                showticklabels=False
            ),
            yaxis=dict(
                showgrid=False,
                zeroline=False,
                showticklabels=False
            ),
            margin=dict(
                l=20,
                r=20,
                t=60,
                b=20
            )
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        # ----------------------------------------------------
        # Centrality
        # ----------------------------------------------------

        st.markdown(
            "### 🔗 Most Connected Products"
        )

        product_nodes = [
            n for n in G.nodes()
            if n.startswith("P_")
        ]

        product_degree = []

        for node in product_nodes:

            product_degree.append(
                {
                    "Product ID": node.replace(
                        "P_",
                        ""
                    ),
                    "Connections": G.degree(node)
                }
            )

        centrality_df = pd.DataFrame(
            product_degree
        )

        if not centrality_df.empty:

            centrality_df = (
                centrality_df
                .sort_values(
                    "Connections",
                    ascending=False
                )
                .head(10)
            )

            fig = px.bar(
                centrality_df,
                x="Product ID",
                y="Connections",
                text_auto=True,
                title="Most Connected Products"
            )

            fig.update_layout(
                template="plotly_dark",
                height=400
            )

            st.plotly_chart(
                fig,
                use_container_width=True
            )

    else:

        st.warning(
            "Not enough customer-product relationships to create the network."
        )


# ============================================================
# DATA PREVIEW
# ============================================================

st.markdown(
    '<div class="section-title">📋 Filtered Dataset</div>',
    unsafe_allow_html=True
)

st.dataframe(
    filtered_df.head(1000),
    use_container_width=True,
    height=400
)


# ============================================================
# DOWNLOAD
# ============================================================

csv_data = filtered_df.to_csv(
    index=False
).encode("utf-8")

st.download_button(
    label="📥 Download Filtered Dataset",
    data=csv_data,
    file_name="filtered_retail_sales.csv",
    mime="text/csv"
)


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "Retail Sales Analytics Dashboard | "
    "Web-Based Graph Analytics"
)